# Графики вибрации по режимам мощности

Источник: `./data/{Месяц}/merged_with_mode.csv`. Строки с пустым `PWR_MODE` не используются.


In [1]:
import re
import os
from dotenv import load_dotenv
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
DATA_DIR = Path("data")
OUT_DIR = Path("charts") / "pwr_mode_vs_vibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS_RU = [
    "Январь", "Февраль", "Март", "Апрель", "Май", "Июнь",
    "Июль", "Август", "Сентябрь", "Октябрь", "Ноябрь", "Декабрь",
]

DATE_COL = "DATE"
TIME_COL = "TIME"
PWR_MODE_COL = "PWR_MODE"

load_dotenv()
vib_cols_raw = os.getenv("VIB_COLS", "")
VIB_COLS = [col.strip() for col in vib_cols_raw.split(",")] if vib_cols_raw else []
VIB_ALIASES = [f"vib_{i:02d}" for i in range(1, len(VIB_COLS) + 1)]
VIB_LABELS = dict(zip(VIB_COLS, VIB_ALIASES))


def vib_label(col: str) -> str:
    return VIB_LABELS.get(col, col)

PWR_MODE_ORDER = [
    "2-100 МВт",
    "100-180 МВт",
    "180-260 МВт",
    "260-340 МВт",
    "340-420 МВт",
    "420-500 МВт",
    "500-580 МВт",
    "580-660 МВт",
    "660-740 МВт",
    "740-820 МВт",
]

sns.set_theme(style="whitegrid")


def _read_month_csv(month: str) -> pd.DataFrame:
    path = DATA_DIR / month / "merged_with_mode.csv"
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path.as_posix()}")
    return pd.read_csv(
        path,
        sep=",",
        decimal=".",
        encoding="utf-8",
        low_memory=False,
        usecols=[DATE_COL, TIME_COL, *VIB_COLS, PWR_MODE_COL],
    )


def _build_timestamp(df: pd.DataFrame) -> pd.Series:
    date_part = (
        df[DATE_COL]
        .astype(str)
        .str.replace("/", ".", regex=False)
        .str.strip()
    )
    time_part = (
        df[TIME_COL]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )
    combined = date_part + " " + time_part
    parsed = pd.to_datetime(combined, format="%d.%m.%Y %H:%M:%S.%f", errors="coerce")
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            combined.loc[missing],
            format="%d.%m.%Y %H:%M:%S",
            errors="coerce",
        )
    return parsed


def _clean_power_mode(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    invalid = cleaned.isna() | cleaned.eq("") | cleaned.str.lower().isin({"nan", "none", "null", "nat"})
    return cleaned.mask(invalid)


def load_year() -> pd.DataFrame:
    frames = []
    for month in MONTHS_RU:
        path = DATA_DIR / month / "merged_with_mode.csv"
        if not path.exists():
            continue
        df = _read_month_csv(month)
        df["timestamp"] = _build_timestamp(df)
        df[PWR_MODE_COL] = _clean_power_mode(df[PWR_MODE_COL])
        df["month"] = month
        df = df.dropna(subset=["timestamp", PWR_MODE_COL])
        frames.append(df)

    if not frames:
        raise RuntimeError("Не найдено ни одного `./data/{Месяц}/merged_with_mode.csv` для загрузки.")

    return pd.concat(frames, ignore_index=True)


def _sanitize_filename(name: str, max_len: int = 180) -> str:
    s = re.sub(r"[<>:\"/\\|?*\n\r\t]", "_", name)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_len:
        s = s[:max_len].rstrip()
    return s


def plot_vib_time_with_pwr_modes(
    df_year: pd.DataFrame,
    vib_cols: list[str],
    style: str = "scatter",  # scatter | line
    outlier_interval: float = 0.99,
) -> pd.DataFrame:
    if not (0.0 < outlier_interval <= 1.0):
        raise ValueError("outlier_interval должен быть в (0; 1]. Например 0.99.")

    needed = ["timestamp", PWR_MODE_COL, *vib_cols]
    missing = [c for c in needed if c not in df_year.columns]
    if missing:
        raise KeyError("В данных отсутствуют столбцы: " + ", ".join(missing))

    df = df_year[needed].copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df[PWR_MODE_COL] = _clean_power_mode(df[PWR_MODE_COL])
    df = df.dropna(subset=["timestamp", PWR_MODE_COL])

    cat = pd.CategoricalDtype(categories=PWR_MODE_ORDER, ordered=True)
    df[PWR_MODE_COL] = df[PWR_MODE_COL].astype(cat)
    df = df.dropna(subset=[PWR_MODE_COL]).sort_values("timestamp")

    palette = sns.color_palette("tab10", n_colors=len(PWR_MODE_ORDER))
    alpha = (1.0 - outlier_interval) / 2.0
    q_low, q_high = alpha, 1.0 - alpha

    rows = []
    for vib_col in vib_cols:
        label = vib_label(vib_col)
        d = df[["timestamp", PWR_MODE_COL, vib_col]].copy()
        d[vib_col] = pd.to_numeric(d[vib_col], errors="coerce")
        d = d.dropna(subset=[vib_col, PWR_MODE_COL])

        before = int(len(d))
        if before == 0:
            rows.append({"vib_col": label, "source_col": vib_col, "rows_before": 0, "rows_after": 0, "dropped": 0, "out": None})
            continue

        qs = (
            d.groupby(PWR_MODE_COL, observed=True)[vib_col]
            .quantile([q_low, q_high])
            .unstack()
            .rename(columns={q_low: "_q_low", q_high: "_q_high"})
        )
        d = d.join(qs, on=PWR_MODE_COL)
        d = d.loc[d[vib_col].between(d["_q_low"], d["_q_high"], inclusive="both")]
        d = d.drop(columns=["_q_low", "_q_high"])

        after = int(len(d))
        dropped = before - after
        if after == 0:
            rows.append({"vib_col": label, "source_col": vib_col, "rows_before": before, "rows_after": 0, "dropped": dropped, "out": None})
            continue

        plt.figure(figsize=(18, 6), dpi=140)
        ax = plt.gca()
        plot_kwargs = dict(
            data=d,
            x="timestamp",
            y=vib_col,
            hue=PWR_MODE_COL,
            hue_order=PWR_MODE_ORDER,
            palette=palette,
            ax=ax,
        )
        if style == "line":
            sns.lineplot(**plot_kwargs, linewidth=0.8, alpha=0.9)
        else:
            sns.scatterplot(**plot_kwargs, s=10, alpha=0.35, linewidth=0)

        ax.set_title(f"Вибрация во времени по режимам мощности\n{label}")
        ax.set_xlabel("Время")
        ax.set_ylabel(label)
        ax.grid(True, which="major", linewidth=0.5, alpha=0.35)

        leg = ax.legend(title=PWR_MODE_COL, loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
        if leg is not None:
            for handle in leg.legend_handles:
                try:
                    handle.set_alpha(1.0)
                except Exception:
                    pass

        plt.tight_layout()
        out_file = OUT_DIR / f"{_sanitize_filename(label)}__time_vs_vibration__by_pwr_mode.png"
        plt.savefig(out_file, bbox_inches="tight")
        plt.close()

        rows.append({
            "vib_col": label,
            "source_col": vib_col,
            "rows_before": before,
            "rows_after": after,
            "dropped": dropped,
            "out": out_file.as_posix(),
        })

    return pd.DataFrame(rows)

In [3]:
df_year = load_year()
report = plot_vib_time_with_pwr_modes(df_year, vib_cols=VIB_COLS)
report.sort_values("rows_after", ascending=False).reset_index(drop=True)

,vib_col,source_col,rows_before,rows_after,dropped,out
0,vib_02,NVGRES1.ST.AM.1SB02V002-AI.Q (ВИБР(Г) ПОДШП 2 ...,24395428,24275061,120367,charts/pwr_mode_vs_vibration/vib_02__time_vs_v...
1,vib_03,NVGRES1.ST.AM.1SB03V002-AI.Q (ВИБР(Г) ПОДШП 3 ...,24395428,24273300,122128,charts/pwr_mode_vs_vibration/vib_03__time_vs_v...
2,vib_04,NVGRES1.ST.AM.1SB04V002-AI.Q (ВИБР(Г) ПОДШП 4 ...,24395428,24269182,126246,charts/pwr_mode_vs_vibration/vib_04__time_vs_v...
3,vib_01,NVGRES1.ST.AM.1SB01V002-AI.Q (ВИБР(Г) ПОДШП 1 ...,24395428,24186711,208717,charts/pwr_mode_vs_vibration/vib_01__time_vs_v...
4,vib_10,NVGRES1.ST.AM.1SB10V002-AI.Q (ВИБР(Г) ПОДШП 10...,24395428,24153709,241719,charts/pwr_mode_vs_vibration/vib_10__time_vs_v...
5,vib_12,NVGRES1.ST.AM.1SP12V002-AI.Q (ВИБР(Г) ПОДШ 12 ...,24395428,24152947,242481,charts/pwr_mode_vs_vibration/vib_12__time_vs_v...
6,vib_05,NVGRES1.ST.AM.1SB05V002-AI.Q (ВИБР(Г) ПОДШП 5 ...,24395428,24152795,242633,charts/pwr_mode_vs_vibration/vib_05__time_vs_v...
7,vib_13,NVGRES1.ST.AM.1SP13V002-AI.Q (ВИБР(Г) ПОДШ 13 ...,24395428,24152176,243252,charts/pwr_mode_vs_vibration/vib_13__time_vs_v...
8,vib_08,NVGRES1.ST.AM.1SB08V002-AI.Q (ВИБР(Г) ПОДШП 8 ...,24395428,24152086,243342,charts/pwr_mode_vs_vibration/vib_08__time_vs_v...
9,vib_09,NVGRES1.ST.AM.1SB09V002-AI.Q (ВИБР(Г) ПОДШП 9 ...,24395428,24152012,243416,charts/pwr_mode_vs_vibration/vib_09__time_vs_v...
